# Module 2 — Place Cells & Rate Maps

A place cell encodes *where* the animal is. This module is about building a population of them, reading their firing at a single point and over the whole environment, driving them along a trajectory to get a `(T, n)` rate matrix, and checking the salient properties before shipping the data off to another object.


> **Place Cells**: A hippocampal place cell fires in a compact patch of the environment, its **place field** (O'Keefe & Dostrovsky, 1971). In an open arena that firing is largely independent of which way the animal faces, but the same cells become directional on linear tracks (Muller et al., 1994, McNaughton et al., 1983). RatInABox models a field as a parameterized bump centered at a fixed location and evaluated at the agent's current position (George et al., 2024).

## Outcomes from this Module

- Instantiate `PlaceCells` on an `Agent` and set `n`, `description`, `widths`, `place_cell_centres`
- Read firing with `get_state()` and learn the `(n, n_positions)` **shape convention**
- Build **analytic rate maps** with `get_state(evaluate_at="all")` and line them up with `env.flattened_discrete_coords`
- Compare the five field `description`s: `gaussian`, `gaussian_threshold`, `diff_of_gaussians`, `top_hat`, `one_hot`
- Control field size with `widths`, and firing range with `max_fr` / `min_fr`
- Place centres **by hand** with `place_cell_centres` to tile a 1-D track evenly
- Collect the `(T, n)` **firingrate** matrix by looping `Ag.update()` then `pc.update()`
- Add temporally correlated rate noise with `noise_std` and `noise_coherence_time`
- Read out a 1-D population with `mountain_plot`
- **Check** that fields are non-directional by binning firing over head direction
- Know that `wall_geometry` exists and what it controls

## Tutorial

### 0. RiaB Boilerplate

Same setup as module 1. The only new import is `PlaceCells` from `ratinabox.Neurons`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ratinabox
from ratinabox.Environment import Environment
from ratinabox.Agent import Agent
from ratinabox.Neurons import PlaceCells

ratinabox.autosave_plots = False
np.random.seed(0)               # for reproducibility
rng = np.random.default_rng(0)  # for sampling

### 1. An arena, an agent, and a population of place cells

`PlaceCells` takes the **`Agent` as its first positional argument**. The cells hold a reference to the agent and read its position on every call, so you never hand them coordinates by hand. The params that matter most are `n` (how many cells), `description` (the field shape), and `widths` (the field size in meters).

In [ ]:
env = Environment(params={"scale": 1.0})    # 1 m x 1 m open arena
Ag = Agent(env, params={"dt": 0.05})        # OU locomotion, as in module 1

pc = PlaceCells(Ag, params={
    "n": 20,                      # 20 cells
    "description": "gaussian",    # smooth bump-shaped field
    "widths": 0.2,                # field size in metres
})
print("n place cells:", pc.n)

By default the 20 centres are scattered at random valid positions in the environment. `pc.place_cell_centres` is an `(n, 2)` array in 2-D, and `pc.plot_place_cell_locations()` draws them on the arena.

In [ ]:
print("place_cell_centres ->", pc.place_cell_centres.shape)
print(np.round(pc.place_cell_centres[:5], 3))

pc.plot_place_cell_locations()
plt.show()

### 2. Reading firing rates

`get_state()` returns firing rates with shape `(n, n_positions)`. Called with no arguments it evaluates at the agent's *current* position, so `n_positions` is 1 and you get back `(n, 1)`.

In [ ]:
s = pc.get_state()
print("get_state() ->", s.shape)     # (20, 1), NOT (20,)

vec = s[:, 0]                        # squeeze to a population vector
print("population vector ->", vec.shape)
print("firing at the current position:", np.round(vec[:5], 3), "...")

> **Gotcha:** the trailing dimension is real. `get_state()` at the current position gives `(n, 1)`, not `(n,)`. If a downstream shape is off by one axis, this is almost always why. Squeeze it with `[:, 0]` or `.reshape(-1)`.

### 3. Rate maps over the whole environment

Pass `evaluate_at="all"` and every cell gets evaluated on the environment's discretized grid instead of at one point. The columns of that matrix line up row-for-row with `env.flattened_discrete_coords`, so you always know which position each column belongs to. This is the **analytic** rate map: the field function evaluated on a grid, independent of where the agent has actually been. Binning real firing over position to get a measured rate map is the standard way place-cell data is summarized (Muller et al., 1987), and we rebuild one from simulated data in problem 2.5.

In [ ]:
rm = pc.get_state(evaluate_at="all")
print("rate map                    ->", rm.shape)                     # (20, 10000)
print("env.flattened_discrete_coords ->", env.flattened_discrete_coords.shape)
print("env.dx =", env.dx, "m, so a 1 m x 1 m arena is a 100 x 100 grid")

For a quick look, let RiaB draw the maps. `chosen_neurons` is either a **string count** (`"5"` means the first five) or an explicit list of indices.

In [ ]:
pc.plot_rate_map(chosen_neurons="5")
plt.show()

You can also evaluate the fields at **arbitrary** positions you supply, which is what you want when comparing against an empirical map computed on a coarser grid.

In [ ]:
pts = np.array([[0.5, 0.5], [0.1, 0.1]])
print("get_state(evaluate_at=None, pos=pts) ->",
      pc.get_state(evaluate_at=None, pos=pts).shape)   # (20, 2)

### 4. Field shapes

`description` sets the field profile. There are five, and all of them build from the same params. Put one cell of each at the same centre and print the min and max of its analytic rate map.

In [ ]:
centre = np.array([[0.5, 0.5]])
descriptions = ["gaussian", "gaussian_threshold", "diff_of_gaussians", "top_hat", "one_hot"]

for d in descriptions:
    cell = PlaceCells(Ag, params={"n": 1, "widths": 0.2, "description": d,
                                  "place_cell_centres": centre})
    m = cell.get_state(evaluate_at="all")[0]
    print(f"{d:19s} min={m.min():+.3f}  max={m.max():+.3f}")

Read that table carefully. `diff_of_gaussians` goes **negative**, the others do not.

- `gaussian`: a smooth bump, strictly positive, never quite reaching zero. The default and the workhorse.
- `gaussian_threshold`: the same bump truncated to exactly zero beyond a radius, so the field has compact support.
- `diff_of_gaussians`: a centre bump with an inhibitory surround. The map dips below zero just outside the field, which is why the min is negative.
- `top_hat`: a flat disc. Rate is `max_fr` inside the field radius and `0` outside, nothing in between.
- `one_hot`: only the single nearest cell is active at any position, a hard tiling. With `n=1` that is degenerate (one cell is always the nearest), which is why its min and max are both 1.

Side by side:

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(11, 3.2))
for ax, d in zip(axs, ["gaussian", "diff_of_gaussians", "top_hat", "one_hot", "gaussian_threshold"]):
    cell = PlaceCells(Ag, params={"n": 1, "widths": 0.2, "description": d,
                                  "place_cell_centres": centre})
    cell.plot_rate_map(chosen_neurons="1", fig=fig, ax=ax)
    ax.set_title(d)
plt.tight_layout()
plt.show()

### 5. Field size

`widths` sets the field size in metres. It takes a scalar (every cell the same) or an array of length `n` (a size per cell). Small fields tile the arena finely and overlap very little, broad fields overlap a lot. A per-cell array is the honest option, because real field size is not a single number: it grows systematically along the dorsoventral axis of hippocampus, from tens of centimetres dorsally to several metres ventrally (Kjelstrup et al., 2008).

In [ ]:
tight = PlaceCells(Ag, params={"n": 20, "widths": 0.05})   # tight fields
broad = PlaceCells(Ag, params={"n": 20, "widths": 0.40})   # broad, overlapping fields

fig, axs = plt.subplots(1, 2, figsize=(8, 3.4))
tight.plot_rate_map(chosen_neurons="1", fig=fig, ax=axs[0])
broad.plot_rate_map(chosen_neurons="1", fig=fig, ax=axs[1])
axs[0].set_title("widths = 0.05")
axs[1].set_title("widths = 0.40")
plt.tight_layout()
plt.show()

# a size per cell, drawn from a distribution
mixed = PlaceCells(Ag, params={"n": 20, "widths": rng.uniform(0.05, 0.4, size=20)})
print("per-cell widths ->", np.round(np.asarray(mixed.widths).reshape(-1)[:5], 3), "...")

`max_fr` and `min_fr` linearly scale the firing range and default to `1` and `0`, so out of the box a rate map peaks at 1 rather than at anything with units. That is fine while you are only looking at shapes. It stops being fine the moment you convert rates to spikes, because the spike draw uses the actual rate, so set `max_fr` to a realistic peak first. In-field peak rates of hippocampal complex-spike cells run from a few Hz up to the low tens of Hz (Muller et al., 1987).

In [ ]:
scaled = PlaceCells(Ag, params={"n": 5, "widths": 0.2, "max_fr": 12.0, "min_fr": 0.2})
m = scaled.get_state(evaluate_at="all")
print(f"max_fr=12, min_fr=0.2  ->  rate map runs {m.min():.2f} to {m.max():.2f} Hz")

### 6. Placing fields by hand to tile a track

Random centres are fine for a demo, but often you want the fields in known places, for example evenly spaced along a linear track so the population tiles it. Pass `place_cell_centres` as an array and RiaB uses exactly those. In a 1-D environment the centres are an `(n, 1)` array and are stored that way.

In [ ]:
track = Environment(params={"dimensionality": "1D", "scale": 2.0})   # a 2 m track
Ag_track = Agent(track, params={"dt": 0.05})

centres = np.linspace(0.1, 1.9, 10).reshape(-1, 1)        # (10, 1)
pc_track = PlaceCells(Ag_track, params={
    "n": 10,
    "widths": 0.2,
    "place_cell_centres": centres,
})
print("place_cell_centres ->", pc_track.place_cell_centres.shape)     # (10, 1)
print("analytic rate map  ->", pc_track.get_state(evaluate_at="all").shape)
print("grid coords        ->", track.flattened_discrete_coords.shape) # (200, 1)

> **Gotcha:** a 1-D `Environment` with solid boundaries and a non-zero `speed_mean` makes RiaB print a `UserWarning` about the agent driving into an end wall. It is harmless here, and module 1 covered the fix (`speed_mean=0` with a non-zero `speed_std`, or an imported trajectory).

To view a 1-D population, `ratinabox.utils.mountain_plot` stacks the tuning curves. Note that `flattened_discrete_coords` is `(n_pos, 1)` in 1-D, so take column 0, and sort by it so the x-axis is monotonic.

In [ ]:
from ratinabox.utils import mountain_plot

x = np.asarray(track.flattened_discrete_coords)[:, 0]     # (200,) take column 0
order = np.argsort(x)                                     # make the x-axis monotonic
ratemaps = pc_track.get_state(evaluate_at="all")[:, order]

fig, ax = mountain_plot(x[order], ratemaps,
                        xlabel="track position (m)", ylabel="place cell")
plt.show()

print("mountain_plot input ->", ratemaps.shape)
print("peak positions (m)  ->", np.round(x[order][ratemaps.argmax(axis=1)], 2))

### 7. Firing over time, the `(T, n)` matrix

Rate maps are static. What you usually want is firing **as the agent moves**. Call `Ag.update()` and then `pc.update()` on every step. Both append to their own histories, and both histories are row-aligned because they share the same clock. Extract with `get_history_arrays()`, same as the agent in module 1.

In [ ]:
n_steps = int(60 / 0.05)      # 60 s at dt = 0.05 -> 1200 steps
for _ in range(n_steps):
    Ag.update()
    pc.update()

nh = pc.get_history_arrays()
for key in ("t", "firingrate", "spikes"):
    print(f"{key:12s} {np.asarray(nh[key]).shape}")

print("\npositions (row-aligned) ->", Ag.get_history_arrays()["pos"].shape)

`nh["firingrate"]` is the `(T, n)` block you would feed a model, one row per timestep and one column per cell. `nh["spikes"]` is a boolean spike train drawn from that rate, at most one spike per cell per `dt`, which we come back to in a later module. A quick visual check of a few cells:

In [ ]:
pc.plot_rate_timeseries(chosen_neurons="5")
plt.show()

Two base params on **any** `Neurons` class add temporally correlated noise to the rates: `noise_std` (the amplitude) and `noise_coherence_time` (how slowly it drifts). `noise_std` is 0 by default, which means no noise at all, so rates are clean unless you ask otherwise. `noise_coherence_time` only does anything once `noise_std` is non-zero.

In [ ]:
noisy = PlaceCells(Ag, params={"n": 20, "widths": 0.2,
                               "noise_std": 0.05, "noise_coherence_time": 0.5})

print("noisy   : noise_std =", noisy.noise_std,
      " noise_coherence_time =", noisy.noise_coherence_time)
print("default : noise_std =", pc.noise_std,
      "    noise_coherence_time =", pc.noise_coherence_time)

### 8. Fields near walls

One heads-up before you meet structured environments. `wall_geometry` controls how a field decays around a barrier, and takes `"geodesic"` (distance measured *around* walls, the default and the physically sensible choice), `"euclidean"` (straight-line distance, so a field bleeds through a wall), or `"line_of_sight"`. In an open arena with no walls all three are identical.

In [ ]:
for wg in ["geodesic", "euclidean", "line_of_sight"]:
    cell = PlaceCells(Ag, params={"n": 1, "widths": 0.2, "wall_geometry": wg,
                                  "place_cell_centres": centre})
    m = cell.get_state(evaluate_at="all")[0]
    print(f"{wg:15s} max={m.max():.4f}  mean={m.mean():.4f}")
print("\nidentical in an open arena, the choice only bites once there are barriers")

Mazes, walls, and what `wall_geometry` actually does to a field get a proper treatment in a later module. For now just know the knob is there and that the default is the one you want.

---

## Key API

| Task | Call |
|---|---|
| Place-cell population | `PlaceCells(Ag, params={"n":20, "description":"gaussian", "widths":0.2})` |
| Field shapes | `description`: `"gaussian"`, `"gaussian_threshold"`, `"diff_of_gaussians"`, `"top_hat"`, `"one_hot"` |
| Field size | `widths` (scalar or length-`n` array, metres) |
| Firing range | `max_fr`, `min_fr` (defaults `1` and `0`) |
| Centres by hand | `params={"place_cell_centres": np.linspace(0.1,1.9,10).reshape(-1,1)}` |
| Inspect centres | `pc.place_cell_centres` → `(n, 2)` in 2-D, `(n, 1)` in 1-D |
| Firing now | `pc.get_state()` → `(n, 1)`, squeeze with `[:, 0]` |
| Analytic rate map | `pc.get_state(evaluate_at="all")` → `(n, n_positions)` |
| Matching positions | `env.flattened_discrete_coords` → `(n_positions, 2)` (`(n_positions, 1)` in 1-D) |
| Firing at given points | `pc.get_state(evaluate_at=None, pos=P)` → `(n, len(P))` |
| Step the cells | `Ag.update()` then `pc.update()` (loop it) |
| Extract a run | `pc.get_history_arrays()` → `{"t":(T,), "firingrate":(T,n), "spikes":(T,n)}` |
| Rate noise | `params={"noise_std":0.05, "noise_coherence_time":0.5}` (`noise_std=0` means no noise) |
| Walls | `params={"wall_geometry": "geodesic" / "euclidean" / "line_of_sight"}` |
| Plots | `pc.plot_place_cell_locations()`, `pc.plot_rate_map(chosen_neurons="5")`, `pc.plot_rate_timeseries(chosen_neurons="5")` |
| 1-D population | `from ratinabox.utils import mountain_plot` then `mountain_plot(X, NbyX, xlabel=..., ylabel=...)` |
| Heading angle | `np.arctan2(hd[:,1], hd[:,0])` from `Ag.get_history_arrays()["head_direction"]` |

---

## Problems

Answers are folded below each problem. Try it yourself first.

### Problem 2.1

Create 20 gaussian place cells with `widths=0.2` in the 1 m x 1 m arena and plot the rate maps for 5 of them.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 2.1"
#| tags: [solution]
np.random.seed(0)

env_21 = Environment(params={"scale": 1.0})
Ag_21 = Agent(env_21, params={"dt": 0.05})
pc_21 = PlaceCells(Ag_21, params={"n": 20, "description": "gaussian", "widths": 0.2})

fig, ax = pc_21.plot_rate_map(chosen_neurons="5")     # first 5 cells
plt.show()

rm_21 = pc_21.get_state(evaluate_at="all")            # analytic map, (n, n_positions)

print("n place cells         :", pc_21.n)
print("rate-map shape (n,pos):", rm_21.shape)
print("grid coords           :", env_21.flattened_discrete_coords.shape)
print("first 5 field centres :\n", np.round(pc_21.place_cell_centres[:5], 3))
print("columns match the grid:",
      bool(rm_21.shape[1] == env_21.flattened_discrete_coords.shape[0]))

### Problem 2.2

Compare field shapes: `gaussian` vs `diff_of_gaussians` vs `top_hat`. Plot one cell's rate map under each and describe the qualitative differences.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 2.2"
#| tags: [solution]
np.random.seed(0)

env_22 = Environment(params={"scale": 1.0})
Ag_22 = Agent(env_22, params={"dt": 0.05})
centre_22 = np.array([[0.5, 0.5]])                    # same centre for all three

shapes = ["gaussian", "diff_of_gaussians", "top_hat"]
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2))
stats = {}

for ax, d in zip(axs, shapes):
    cell = PlaceCells(Ag_22, params={"n": 1, "widths": 0.2, "description": d,
                                     "place_cell_centres": centre_22})
    m = cell.get_state(evaluate_at="all")[0]
    stats[d] = (float(m.min()), float(m.max()))
    cell.plot_rate_map(chosen_neurons="1", fig=fig, ax=ax)
    ax.set_title(d)

plt.tight_layout()
plt.show()

for d in shapes:
    lo, hi = stats[d]
    print(f"{d:19s} min={lo:+.3f}  max={hi:+.3f}")

# a cross-section through the centre makes the profiles obvious
xs = np.linspace(0, 1, 201)
line = np.stack([xs, np.full_like(xs, 0.5)], axis=1)   # y = 0.5

fig, ax = plt.subplots(figsize=(7, 3))
for d in shapes:
    cell = PlaceCells(Ag_22, params={"n": 1, "widths": 0.2, "description": d,
                                     "place_cell_centres": centre_22})
    ax.plot(xs, cell.get_state(evaluate_at=None, pos=line)[0], label=d)
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("x (m), cutting through y = 0.5")
ax.set_ylabel("firing rate")
ax.legend()
plt.tight_layout()
plt.show()

print()
print("gaussian          : smooth bump, strictly positive, decays but never hits 0.")
print("diff_of_gaussians : bump plus an inhibitory surround, so the map goes NEGATIVE")
print(f"                    just outside the field (min = {stats['diff_of_gaussians'][0]:+.3f}).")
print("top_hat           : flat disc, exactly max_fr inside the radius and 0 outside,")
print("                    with a hard edge and no gradient to follow.")

### Problem 2.3

Sweep field size over `widths` in `{0.05, 0.1, 0.2, 0.4}`. For a fixed centre, measure the area where the rate is above half its maximum, and plot area against width.

**Hint:** put one cell at a known centre with `place_cell_centres`, get the analytic map with `evaluate_at="all"`, and remember that each grid square has area `env.dx * env.dx`.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 2.3"
#| tags: [solution]
np.random.seed(0)

env_23 = Environment(params={"scale": 1.0, "dx": 0.01})
Ag_23 = Agent(env_23, params={"dt": 0.05})
centre_23 = np.array([[0.5, 0.5]])          # fixed centre, mid-arena so nothing is clipped
dx = env_23.dx
pixel_area = dx * dx                        # area of one grid square, m^2

widths = [0.05, 0.1, 0.2, 0.4]
areas = []

for w in widths:
    cell = PlaceCells(Ag_23, params={"n": 1, "widths": w,
                                     "place_cell_centres": centre_23})
    m = cell.get_state(evaluate_at="all")[0]
    above_half = m > 0.5 * m.max()
    area = float(above_half.sum() * pixel_area)
    areas.append(area)
    print(f"width = {w:.2f} m  ->  area above half-max = {area:.4f} m^2")

areas = np.array(areas)
widths = np.array(widths)

fig, axs = plt.subplots(1, 2, figsize=(9, 3.4))
axs[0].plot(widths, areas, "o-")
axs[0].set_xlabel("widths (m)")
axs[0].set_ylabel("area above half-max (m$^2$)")
axs[0].set_title("linear axes")

axs[1].loglog(widths, areas, "o-", label="measured")
axs[1].loglog(widths, areas[0] * (widths / widths[0]) ** 2, "k--", label="slope 2")
axs[1].set_xlabel("widths (m)")
axs[1].set_ylabel("area above half-max (m$^2$)")
axs[1].set_title("log-log")
axs[1].legend()
plt.tight_layout()
plt.show()

# fit the exponent: a 2-D bump should scale as width^2
slope = np.polyfit(np.log(widths), np.log(areas), 1)[0]
print("\nareas (m^2):", np.round(areas, 4))
print("monotonically increasing:", bool(np.all(np.diff(areas) > 0)))
print(f"log-log slope = {slope:.2f}  (2 means area grows with the SQUARE of width)")
print("so doubling widths roughly quadruples the field area")

### Problem 2.4

Place centres by hand to tile a 2 m 1-D track evenly. Run the agent and show the population with `mountain_plot`.

**Hint:** in 1-D the grid coords are `(n_pos, 1)`, so sort by column 0 before plotting.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 2.4"
#| tags: [solution]
np.random.seed(0)

track_24 = Environment(params={"dimensionality": "1D", "scale": 2.0})
# speed_mean = 0 with a healthy speed_std gives back-and-forth running instead of the
# agent parking against an end wall (module 1, section 6).
Ag_24 = Agent(track_24, params={"dt": 0.05, "speed_mean": 0.0, "speed_std": 0.2})

n_cells = 10
centres_24 = np.linspace(0.1, 1.9, n_cells).reshape(-1, 1)     # (10, 1), evenly spaced
pc_24 = PlaceCells(Ag_24, params={"n": n_cells, "widths": 0.2,
                                  "place_cell_centres": centres_24})

for _ in range(int(120 / 0.05)):        # 2 minutes of running
    Ag_24.update()
    pc_24.update()

# analytic rate map, sorted so the track axis is monotonic
x = np.asarray(track_24.flattened_discrete_coords)[:, 0]       # column 0, the hint
order = np.argsort(x)
ratemaps = pc_24.get_state(evaluate_at="all")[:, order]         # (10, n_pos)

fig, ax = mountain_plot(x[order], ratemaps,
                        xlabel="track position (m)", ylabel="place cell")
plt.show()

# and the same population as it was actually driven by the agent
pc_24.plot_rate_timeseries(chosen_neurons="5")
plt.show()

peaks = x[order][ratemaps.argmax(axis=1)]
h24 = pc_24.get_history_arrays()

print("place_cell_centres  :", pc_24.place_cell_centres.shape)
print("rate map (n, n_pos) :", ratemaps.shape)
print("requested centres   :", np.round(centres_24[:, 0], 2))
print("measured peaks      :", np.round(peaks, 2))
print("max |peak - centre| :", f"{np.abs(peaks - centres_24[:, 0]).max():.3f} m")
print("firingrate matrix   :", h24["firingrate"].shape)

### Problem 2.5

Run 10 minutes, extract the `(T, n)` firingrate matrix, and rebuild an **empirical** rate map by binning firing over position. Compare it to the analytic map.

**Hint:** `np.histogram2d` weighted by the cell's rate, divided by the occupancy histogram, gives the mean rate per bin. Watch for empty bins.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 2.5"
#| tags: [solution]
np.random.seed(0)

env_25 = Environment(params={"scale": 1.0})
Ag_25 = Agent(env_25, params={"dt": 0.05})
pc_25 = PlaceCells(Ag_25, params={"n": 20, "description": "gaussian", "widths": 0.2})

n_steps = int(600 / 0.05)               # 10 minutes at dt = 0.05 -> 12000 steps
for _ in range(n_steps):
    Ag_25.update()
    pc_25.update()

fr = pc_25.get_history_arrays()["firingrate"]     # (T, 20)
pos = Ag_25.get_history_arrays()["pos"]           # (T, 2), row-aligned to fr
print("firingrate matrix :", fr.shape)
print("positions         :", pos.shape)
print("rows aligned      :", bool(fr.shape[0] == pos.shape[0]))

cell = 0
nb = 20
edges = np.linspace(0.0, 1.0, nb + 1)
ctr = 0.5 * (edges[:-1] + edges[1:])

# occupancy (samples per bin) and total firing per bin, then divide
occ, _, _ = np.histogram2d(pos[:, 0], pos[:, 1], bins=[edges, edges])
tot, _, _ = np.histogram2d(pos[:, 0], pos[:, 1], bins=[edges, edges], weights=fr[:, cell])
empirical = np.where(occ > 0, tot / np.maximum(occ, 1), np.nan)     # empty bins -> nan

# the analytic field evaluated at exactly the same bin centres
XX, YY = np.meshgrid(ctr, ctr, indexing="ij")
grid = np.stack([XX.ravel(), YY.ravel()], axis=1)                  # (400, 2)
analytic = pc_25.get_state(evaluate_at=None, pos=grid)[cell].reshape(nb, nb)

visited = ~np.isnan(empirical)
r = float(np.corrcoef(empirical[visited], analytic[visited])[0, 1])
err = float(np.nanmax(np.abs(empirical - analytic)))

fig, axs = plt.subplots(1, 3, figsize=(11, 3.2))
# .T + origin="lower" because histogram2d indexes [x, y] and imshow wants [row, col]
im0 = axs[0].imshow(empirical.T, origin="lower", extent=[0, 1, 0, 1])
axs[0].set_title(f"empirical (cell {cell})")
im1 = axs[1].imshow(analytic.T, origin="lower", extent=[0, 1, 0, 1])
axs[1].set_title("analytic")
im2 = axs[2].imshow(np.abs(empirical - analytic).T, origin="lower", extent=[0, 1, 0, 1])
axs[2].set_title("|difference|")
for im, ax in zip((im0, im1, im2), axs):
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

print(f"\nvisited bins            : {int(visited.sum())} / {nb * nb}")
print(f"empirical vs analytic r : {r:.4f}")
print(f"largest bin difference  : {err:.4f}")
print("The rate is a deterministic function of position, so the two agree almost exactly.")
print("The only limit is coverage: unvisited bins stay empty, which is a property of the")
print("trajectory, not of the cell. Bin finer or run longer and the gaps shrink.")

### Problem 2.6

Show that a place cell's firing does not depend on head direction in the open field. Real place cells are largely non-directional in open arenas too (Muller et al., 1994). Bin firing by head direction and confirm the curve is close to flat.

**Hint:** `head_direction` is a `(T, 2)` unit vector, so use `np.arctan2` to get an angle. Report the coefficient of variation across bins, and remember that any residual bumpiness is uneven sampling rather than real tuning.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution 2.6"
#| tags: [solution]
np.random.seed(0)

env_26 = Environment(params={"scale": 1.0})
Ag_26 = Agent(env_26, params={"dt": 0.05})
pc_26 = PlaceCells(Ag_26, params={"n": 20, "description": "gaussian", "widths": 0.2})

for _ in range(int(600 / 0.05)):        # 10 minutes
    Ag_26.update()
    pc_26.update()

fr = pc_26.get_history_arrays()["firingrate"]        # (T, 20)
hd = Ag_26.get_history_arrays()["head_direction"]    # (T, 2) unit vectors
angle = np.arctan2(hd[:, 1], hd[:, 0])               # (T,) radians in (-pi, pi]

nb = 12
edges = np.linspace(-np.pi, np.pi, nb + 1)
ang_ctr = 0.5 * (edges[:-1] + edges[1:])
b = np.clip(np.digitize(angle, edges) - 1, 0, nb - 1)

def hd_tuning(rates):
    # mean firing in each head-direction bin
    return np.array([rates[b == k].mean() for k in range(nb)])

cell = 0
tuning = hd_tuning(fr[:, cell])
cv = float(tuning.std() / tuning.mean())     # coefficient of variation across HD bins

# a genuinely directional signal for comparison: cos(angle - preferred), rectified
preferred = 0.7
fake_directional = np.clip(np.cos(angle - preferred), 0, None)
tuning_dir = hd_tuning(fake_directional)
cv_dir = float(tuning_dir.std() / tuning_dir.mean())

fig, axs = plt.subplots(1, 2, figsize=(9, 4), subplot_kw={"projection": "polar"})
for ax, tc, name, c in ((axs[0], tuning, f"place cell {cell}", "C0"),
                        (axs[1], tuning_dir, "a directional signal", "C3")):
    ax.plot(np.append(ang_ctr, ang_ctr[0]), np.append(tc, tc[0]), color=c)
    ax.fill(np.append(ang_ctr, ang_ctr[0]), np.append(tc, tc[0]), color=c, alpha=0.2)
    ax.set_ylim(0, None)
    ax.set_title(name, pad=18)
plt.tight_layout()
plt.show()

print("per-HD-bin mean firing (cell 0):", np.round(tuning, 3))
print(f"CV across HD bins, place cell       : {cv:.3f}")
print(f"CV across HD bins, directional cell : {cv_dir:.3f}")

# check it holds across the whole population, not just one cell
cvs = np.array([hd_tuning(fr[:, i]).std() / hd_tuning(fr[:, i]).mean()
                for i in range(pc_26.n)])
print(f"\npopulation CV: min {cvs.min():.3f}, median {np.median(cvs):.3f}, max {cvs.max():.3f}")
print("Flat. The field is a function of position only, so head direction cannot enter it.")
print("The small residual is uneven sampling (the animal does not visit every location")
print("facing every direction equally), not directional tuning.")

# References

@article{george2024ratinabox,
  author  = {George, Tom M. and Rastogi, Mehul and de Cothi, William and
             Clopath, Claudia and Stachenfeld, Kimberly and Barry, Caswell},
  title   = {{RatInABox}, a toolkit for modelling locomotion and neuronal
             activity in continuous environments},
  journal = {eLife},
  volume  = {13},
  pages   = {e85274},
  year    = {2024},
  doi     = {10.7554/eLife.85274}
}

@article{okeefe1971hippocampus,
  author  = {O'Keefe, J. and Dostrovsky, J.},
  title   = {The hippocampus as a spatial map. Preliminary evidence from unit
             activity in the freely-moving rat},
  journal = {Brain Research},
  volume  = {34},
  number  = {1},
  pages   = {171--175},
  year    = {1971},
  doi     = {10.1016/0006-8993(71)90358-1}
}

@article{muller1987spatial,
  author  = {Muller, Robert U. and Kubie, John L. and Ranck, James B.},
  title   = {Spatial firing patterns of hippocampal complex-spike cells in a
             fixed environment},
  journal = {The Journal of Neuroscience},
  volume  = {7},
  number  = {7},
  pages   = {1935--1950},
  year    = {1987},
  doi     = {10.1523/JNEUROSCI.07-07-01935.1987}
}

@article{muller1994directional,
  author  = {Muller, Robert U. and Bostock, Elizabeth and Taube, Jeffrey S. and
             Kubie, John L.},
  title   = {On the directional firing properties of hippocampal place cells},
  journal = {The Journal of Neuroscience},
  volume  = {14},
  number  = {12},
  pages   = {7235--7251},
  year    = {1994},
  doi     = {10.1523/JNEUROSCI.14-12-07235.1994}
}

@article{mcnaughton1983contributions,
  author  = {McNaughton, B. L. and Barnes, C. A. and O'Keefe, J.},
  title   = {The contributions of position, direction, and velocity to single
             unit activity in the hippocampus of freely-moving rats},
  journal = {Experimental Brain Research},
  volume  = {52},
  number  = {1},
  pages   = {41--49},
  year    = {1983},
  doi     = {10.1007/BF00237147}
}

@article{kjelstrup2008finite,
  author  = {Kjelstrup, Kirsten B. and Solstad, Trygve and Brun, Vegard H. and
             Hafting, Torkel and Leutgeb, Stefan and Witter, Menno P. and
             Moser, Edvard I. and Moser, May-Britt},
  title   = {Finite scale of spatial representation in the hippocampus},
  journal = {Science},
  volume  = {321},
  number  = {5885},
  pages   = {140--143},
  year    = {2008},
  doi     = {10.1126/science.1157086}
}